# Interacting with vLLM: The User API

Welcome to the first chapter in our journey to understand vLLM! We'll start at the most logical place: the user's point of view. How do you actually *use* vLLM to run inference? The primary entry point for this is a class called the `LLMEngine`.

By the end of this notebook, you will be able to initiate and manage LLM inference requests through a high-level API. You will understand the basic lifecycle of a request, from the moment you submit your prompt to when you get the generated text back. This will lay the groundwork for understanding *how* vLLM processes these requests so efficiently under the hood.

In [ ]:
import dataclasses
import time
import collections
import uuid
from typing import List, Dict, Any

# For our visualization later
import graphviz

### The Core Idea: A Restaurant's Host

Imagine you're going to a very popular, very busy restaurant. You don't just walk into the kitchen and tell the chef what you want. Instead, you talk to the host at the front.

This is the role of the `LLMEngine`. It's the public-facing interface, the host of the restaurant.

1.  **You (The User) Arrive**: You come with a request: "I'd like a table for four, and we want to order the special." In vLLM terms, this is your prompt ("What is the capital of France?") and your sampling parameters (e.g., `temperature=0.7`, `max_tokens=50`).

2.  **The Host (`LLMEngine`) Takes Your Order**: The host takes your name, gives you a unique ticket number (`request_id`), and writes down your order. They don't cook the food. Their job is to accept your request, organize it, and pass it on to the kitchen. The method for this is `add_request()`.

3.  **The Kitchen (`EngineCore`) Gets the Order**: The host places your order slip in a queue for the kitchen. The kitchen staff (the `Scheduler` and `Executor`, which we'll cover later) will pick it up when they are ready.

4.  **Checking for Your Food**: You don't repeatedly ask the host "Is it ready yet?". Instead, the host periodically checks the kitchen's service window to see which orders are complete. In vLLM, this "checking" action is done by calling the `step()` method. When `step()` is called, the `LLMEngine` polls the backend for any finished results.

This separation is crucial. The host can take many orders very quickly, letting them pile up for the kitchen. The kitchen can then grab a whole batch of similar orders (e.g., all the steak orders) and cook them together efficiently. This is the essence of batching, which is key to vLLM's speed. The `LLMEngine` is the component that enables this by decoupling request submission from execution.

Let's build a simplified version of this system.

In [ ]:
# --- Mock Internal Components ---
# These represent the deeper parts of vLLM that LLMEngine talks to.

@dataclasses.dataclass
class InternalRequest:
    """A structured request passed from the LLMEngine to the core."""
    request_id: str
    prompt: str
    output_text: str = ""
    is_finished: bool = False

@dataclasses.dataclass
class RequestOutput:
    """The final output returned to the user."""
    request_id: str
    text: str
    is_finished: bool

class MockEngineCoreClient:
    """A fake version of the EngineCore that LLMEngine communicates with."""
    def __init__(self):
        self.request_queue = collections.deque()
        self.processed_requests: Dict[str, InternalRequest] = {}

    def add_request(self, request: InternalRequest):
        print(f"  [EngineCore] Received request '{request.request_id}'. Adding to queue.")
        self.request_queue.append(request)

    def get_output(self) -> List[InternalRequest]:
        """Simulates one step of processing and returns finished requests."""
        if not self.request_queue:
            return []
        
        # "Process" one request from the queue
        request_to_process = self.request_queue.popleft()
        print(f"  [EngineCore] Processing request '{request_to_process.request_id}'.")
        request_to_process.output_text = f"The capital of France is Paris."
        request_to_process.is_finished = True
        self.processed_requests[request_to_process.request_id] = request_to_process
        
        return [request_to_process]

# --- The User-Facing API ---

class SimpleLLMEngine:
    """The user's main entry point for submitting and retrieving requests."""
    def __init__(self):
        self.engine_core_client = MockEngineCoreClient()
        self.unfinished_requests: Dict[str, InternalRequest] = {}
        print("SimpleLLMEngine initialized.")

    def add_request(self, request_id: str, prompt: str):
        """Accepts a new request from the user and sends it to the core."""
        print(f"[LLMEngine] Adding request '{request_id}' with prompt: '{prompt}'")
        internal_req = InternalRequest(request_id=request_id, prompt=prompt)
        self.unfinished_requests[request_id] = internal_req
        self.engine_core_client.add_request(internal_req)

    def step(self) -> List[RequestOutput]:
        """Performs one processing step and returns any completed outputs."""
        print("\n[LLMEngine] Calling step()...")
        core_outputs = self.engine_core_client.get_output()

        user_outputs = []
        for core_out in core_outputs:
            if core_out.is_finished:
                print(f"[LLMEngine] Request '{core_out.request_id}' is finished.")
                output = RequestOutput(
                    request_id=core_out.request_id,
                    text=core_out.output_text,
                    is_finished=True
                )
                user_outputs.append(output)
                # Clean up finished request
                del self.unfinished_requests[core_out.request_id]
        
        return user_outputs
        
    def has_unfinished_requests(self) -> bool:
        """Checks if there are any pending requests."""
        return len(self.unfinished_requests) > 0

### Walking Through the Implementation

Let's break down our `SimpleLLMEngine`.

-   **`MockEngineCoreClient`**: This is our stand-in for the complex "kitchen" part of vLLM.
    -   It has a `request_queue` to hold incoming orders.
    -   `add_request` simply puts a new order in this queue.
    -   `get_output` simulates doing one step of work. In our simple version, it just takes one request from the queue, "processes" it by filling in a canned response, and returns it. In the real vLLM, this is an incredibly complex operation involving scheduling, memory management, and GPU execution.

-   **`SimpleLLMEngine`**: This is the host, the class a user actually interacts with.
    -   **`__init__`**: It holds a client to communicate with the `EngineCore`. This separation is a key architectural choice, allowing the core to potentially run in a separate process or even on a different machine.
    -   **`add_request(request_id, prompt)`**: This is the public API. It's clean and simple. It takes a user's prompt, packages it into our more structured `InternalRequest` object, and forwards it to the engine core. It also keeps track of which requests are still running in its `unfinished_requests` dictionary.
    -   **`step()`**: This is the main "event loop" function. The `LLMEngine` asks the `EngineCoreClient` if any work has been completed (`get_output`). If so, it processes these completions, transforms them from the internal format to the user-facing `RequestOutput` format, and cleans up its internal state.

The key takeaway is the flow: `User -> add_request() -> LLMEngine -> EngineCoreClient -> Request Queue`. Then, on a separate timeline: `LLMEngine -> step() -> EngineCoreClient -> Get Completed -> User`.

In [ ]:
# --- Demonstration ---

# 1. Initialize the engine
engine = SimpleLLMEngine()

# 2. Add some requests. Notice these calls return immediately.
# The engine doesn't wait for one to finish before accepting the next.
request1_id = f"req-{uuid.uuid4().hex[:4]}"
request2_id = f"req-{uuid.uuid4().hex[:4]}"

engine.add_request(request_id=request1_id, prompt="What is the capital of France?")
engine.add_request(request_id=request2_id, prompt="What is the capital of Germany?")

# 3. Drive the engine's processing loop.
# In a real server, this would be a continuous `while True:` loop.
print("\n--- Starting Inference Loop ---")
while engine.has_unfinished_requests():
    # Each call to step() processes one batch of work.
    request_outputs = engine.step()
    
    if request_outputs:
        for output in request_outputs:
            print(f"\n[User] Received output for request '{output.request_id}':")
            print(f"  -> Text: '{output.text}'")
            print(f"  -> Finished: {output.is_finished}")
    else:
        print("[User] No outputs ready yet in this step.")
        
    # Let's add a small delay to make the steps clearer
    time.sleep(1)

print("\n--- All requests finished! ---")

### Going Deeper: Why Decouple `add_request` and `step`?

In our simple demonstration, it might seem overly complicated to have separate `add_request` and `step` methods. Why not just have a single `generate(prompt)` method that blocks until the result is ready?

This is one of the most important design decisions in vLLM. The separation of submission and execution is what enables **continuous batching**.

-   **High Concurrency**: A web server using vLLM can receive hundreds of requests per second. The `add_request` method is very lightweight; it just adds an item to a queue and returns. This means the server can accept new requests without getting blocked by a long-running generation. It can immediately handle the *next* incoming user connection.

-   **Efficient GPU Utilization**: GPUs are most efficient when they process large batches of data at once. A simple "one in, one out" `generate(prompt)` function would lead to a batch size of 1, which is terribly inefficient. By decoupling, the `LLMEngine` can accumulate many user requests in its queue. When `step()` is called, the `Scheduler` (inside the `EngineCore`) can look at all waiting requests and build an optimal batch to send to the GPU, maximizing throughput.

This design transforms the system from a synchronous, one-at-a-time process into an asynchronous, high-throughput pipeline, which is the cornerstone of vLLM's performance.

In [ ]:
# --- Visualization of the Request Flow ---

dot = graphviz.Digraph('LLMEngine Flow', comment='The lifecycle of a user request')
dot.attr('node', shape='box', style='rounded')

# Define the main components
dot.node('User', 'User')
dot.node('LLMEngine', 'LLMEngine (The Host)')
dot.node('EngineCore', 'EngineCore (The Kitchen)')
dot.node('RequestQueue', 'Internal Request Queue', shape='cylinder')

# Style the nodes
dot.node('User', color='blue')
dot.node('LLMEngine', color='darkgreen')
dot.node('EngineCore', color='darkorange')


# Define the flow for adding a request
with dot.subgraph(name='cluster_submission') as c:
    c.attr(label='1. Request Submission (Asynchronous)', style='dashed')
    c.edge('User', 'LLMEngine', label='add_request(prompt, params)')
    c.edge('LLMEngine', 'RequestQueue', label='Forwards structured\nInternalRequest')

# Define the flow for processing and retrieving results
with dot.subgraph(name='cluster_processing') as c:
    c.attr(label='2. Processing Loop (Synchronous Step)', style='dashed')
    c.edge('RequestQueue', 'EngineCore', label='Pulls batch of requests')
    c.edge('EngineCore', 'LLMEngine', label='step() -> polls for results')
    c.edge('LLMEngine', 'User', label='Returns RequestOutput')

# Display the graph in the notebook
dot

### Summary and What's Next

In this notebook, we peeled back the first layer of vLLM and explored its user-facing API, the `LLMEngine`. We saw how it acts as a sophisticated front desk, decoupling the process of accepting user prompts from the complex work of actually generating text.

**Key Takeaways:**
*   The `LLMEngine` is the primary interface for users, providing methods like `add_request` and `step`.
*   It serves as a facade, hiding the complex internal machinery of the inference engine.
*   The separation of request submission (`add_request`) and result retrieval (`step`) is a deliberate design choice that enables asynchronous operation and continuous batching for maximum throughput.

We've learned *how* to talk to vLLM, but we've treated the backend `EngineCore` as a black box. What actually happens to a request after the `LLMEngine` passes it on?

In the next notebook, **The Heart of vLLM: Request Orchestration with EngineCore**, we will open up that black box and explore the central orchestrator that manages the entire inference lifecycle.